# Этап 0. Таблица заключений с обвинениями `compas_custody_charges` из ProPublica `compas.db`

Ноутбук полностью воспроизводит таблицу `compas_custody_charges.csv`, которая лежит на GitHub
(`zharraya05/compas-process-mining`) вместе со скриптом `build_from_compas_db.py`.
Код здесь тот же, что в скрипте, только разбит на шаги с пояснениями.

**Что получится.** Одна строка = одно обвинение внутри одного заключения (custody) одного человека.

| Правило | Как сделано |
|---|---|
| Кто человек | `id` = `person_id` из базы (не имя) |
| Какие люди | все `person_id`, у которых есть арест в `casearrest` |
| Заключения | `jailhistory` + `prisonhistory`, вход (`in_custody`) не раньше 2013-01-01 |
| Обвинения заключения | арест в ±1 день → иначе преступление за 7 дней до входа → иначе последнее дело за год до входа → иначе пусто |
| Производные столбцы | как в таблице Даши `individual_custody_timeline_rfm.csv` |

База открывается **только на чтение** и не меняется.

В конце ноутбук сверяет результат с опубликованной таблицей (число строк, столбцы, все значения).

## Шаг 0. Настройки

Здесь задаются пути. Поменяйте их, если файлы лежат в другом месте.

- `DB_PATH` — файл ProPublica `compas.db`;
- `OUT_DIR` — куда сохранить результат (по умолчанию папка `out` рядом с ноутбуком);
- `REFERENCE_CSV` — опубликованная таблица для сверки (если файла нет, сверка пропускается);
- `WRITE_XLSX` — записывать ли ещё и Excel (около минуты и 7 МБ).

Окружение: `D:\Git\diploma\.venv` (нужны `pandas`, `numpy`, для Excel — `openpyxl`).

In [5]:
DB_PATH = r"compas.db"
OUT_DIR = "out"
REFERENCE_CSV = r"compas_custody_charges.csv"
WRITE_XLSX = True

START = "2013-01-01"   # заключения раньше этой даты не берём

In [6]:
import os
import re
import sqlite3

import numpy as np
import pandas as pd

print("pandas", pd.__version__, "| numpy", np.__version__)
print("текущая папка:", os.getcwd())
print("база найдена:", os.path.exists(DB_PATH), "|", os.path.abspath(DB_PATH))
START = pd.Timestamp(START)

pandas 2.3.3 | numpy 2.5.3
текущая папка: D:\diploma\jupyter lab
база найдена: True | D:\diploma\jupyter lab\compas.db


## Шаг 1. Читаем нужные таблицы базы

Из `compas.db` нужны пять таблиц (подробное описание всех таблиц — в `COMPAS_DB_TABLES.md`):

| Таблица | Что берём | Зачем |
|---|---|---|
| `people` | `id, age, sex, race` | данные о человеке |
| `casearrest` | `person_id, case_number, arrest_date` | аресты: когда человека задержали по какому делу |
| `charge` | `person_id, case_number, offense_date, charge_number, charge_degree, charge` | обвинения по делам |
| `jailhistory` | `person_id, in_custody, out_custody` | заключения в окружной тюрьме (jail) |
| `prisonhistory` | `person_id, in_custody, out_custody` | заключения в тюрьме штата (prison) |

Обе истории заключений склеиваем в одну таблицу `hist`, а столбец `custody_source` запоминает, откуда строка (`jail` / `prison`).
Соединение открыто в режиме `mode=ro` (только чтение). Путь превращается в URI через `Path.as_uri()`, иначе SQLite не открывает файл, если в пути есть пробел (как в `jupyter lab`) или обратные слэши Windows.

In [7]:
from pathlib import Path

db_file = Path(DB_PATH).expanduser().resolve()   # абсолютный путь (относительный считается от папки ноутбука)
if not db_file.exists():
    raise FileNotFoundError(f"нет файла базы: {db_file}  (текущая папка: {Path.cwd()})")
# as_uri() даёт file:///D:/diploma/jupyter%20lab/compas.db — пробелы и \ в пути Windows так не ломают URI
con = sqlite3.connect(db_file.as_uri() + "?mode=ro", uri=True)

arrests = pd.read_sql("select person_id, case_number, arrest_date from casearrest", con)
people = pd.read_sql("select id as person_id, age, sex, race from people", con)
charges = pd.read_sql("select person_id, case_number, offense_date, charge_number, "
                      "charge_degree, charge from charge", con)
hist = pd.concat([pd.read_sql(f"select person_id, in_custody, out_custody, '{src}' as custody_source "
                              f"from {tbl}", con)
                  for tbl, src in [("jailhistory", "jail"), ("prisonhistory", "prison")]])
con.close()

for name, t in [("casearrest", arrests), ("people", people), ("charge", charges), ("jail+prison", hist)]:
    print(f"{name:12s} {t.shape}")

casearrest   (128183, 3)
people       (11757, 4)
charge       (148086, 6)
jail+prison  (27056, 4)


## Шаг 2. Даты и связка «арест → обвинения»

Даты в базе записаны строками, иногда со временем. Приводим их к дате без времени (`normalize()`),
чтобы сравнивать «по дням». Нераспознанные даты становятся пустыми (`errors="coerce"`).

Обвинения без текста (`charge` пустой) выбрасываем.

`arr_ch` — каждый арест вместе со всеми обвинениями его дела: соединяем `casearrest` и `charge`
по паре (`person_id`, `case_number`). Повторы одного и того же ареста (тот же человек, дело и дата) убираем заранее.

In [8]:
def day(s):
    """строка с датой (и, может быть, временем) -> дата без времени"""
    return pd.to_datetime(s, format="mixed", errors="coerce").dt.normalize()


arrests["arrest_date"] = day(arrests["arrest_date"])
charges["offense_date"] = day(charges["offense_date"])
charges = charges.dropna(subset=["charge"])

arr_ch = arrests.drop_duplicates(["person_id", "case_number", "arrest_date"]).merge(
    charges, on=["person_id", "case_number"])
print("обвинений с текстом:", len(charges), "| пар арест-обвинение:", len(arr_ch))
arr_ch.head()

обвинений с текстом: 147513 | пар арест-обвинение: 162590


,person_id,case_number,arrest_date,offense_date,charge_number,charge_degree,charge
0,533,13005832CF10A,2013-04-23,2013-04-23,1,(F3),Driving While License Revoked
1,533,13005832CF10A,2013-04-23,2013-04-23,2,(M2),Petit Theft
2,533,14007863MM10A,2014-05-13,2014-05-13,1,(M1),Extradition/Defendants
3,900,13015312CF10A,2013-11-03,2013-11-03,1,(F2),Burglary Unoccupied Dwelling
4,900,13015312CF10A,2013-11-03,2013-11-03,2,(M1),Battery


## Шаг 3. Категории преступлений

Каждое обвинение (`c_charge_desc`) относим к одной из категорий по регулярным выражениям.
Правила проверяются **по порядку**, срабатывает первое подошедшее: `NoCharge → Sex → Traffic → Violent → Drug → Weapon → Fraud → Property`.
Не подошло ни одно правило — `Other`; пустое обвинение — `Unknown`.

Это та же группировка (9 категорий), что была в таблице на GitHub. Более подробная группировка на 15 состояний
(ResistPolice, PublicOrder, Justice и т.д.) делается уже на этапе 1, поверх этой таблицы, и здесь не нужна.

In [9]:
# категории преступлений — те же правила, что в stage1/crime_categories.py
RULES = [
    ("NoCharge", r"no charge"),
    ("Sex", r"lewd|lasciv|sex|prostitu|indecent expos|voyeur|porn|obscen|molest|travel.*minor|child exploit|computer for child"),
    ("Traffic", r"\bdui\b|d\.u\.i|\bdwi\b|\bdwls\b|driv|licen|\bdl\b|\blic\b|motorcycle|vehicle insurance|motor veh ins"
                r"|insur.*card|proof of invalid insur|flee|elud|leav.*(scen|acc)|lve/scen|reckless driv|register vehicle"
                r"|tag|sticker|boater|load|tri-rail|taxi"
                # нарушения ПДД (часто идут сопутствующими обвинениями в том же деле)
                r"|safety belt|stbelt|seat ?belt|child restraint|pers/inj/prot|submit insurance"
                r"|stop sign|signal|red light|flashing|yield|ftyrow|\bspeed|too fast|following too closely"
                r"|\blanes?\b|\bturn|backing|stop/stand|stop stand|stand/park|imp park|improp park|park handicap|park/obstruct|hov|sunscr|wndow|windshield|window"
                r"|due care|unsafe/improp|unsafe/improper|improp equip|lights|lamps|mcycle|bike|ped fail|x-walk"
                r"|refuse submit blood|breath test|render aid|rept accd|road surface|sounds>1000|display regist"),
    ("Violent", r"murder|manslaughter|homicide|home invasion|kidnap|carjack|robbery|battery|\bbatt\b|batt/|assault|false imprison|stalk"
                r"|child abuse|cruelty|neglect|abuse|shoot|throw.*missile|throw in occup|harm public servant"
                r"|injunc|dom vi|domestic|dating viol|threat|interference with custody|culpable negligence"
                r"|exposes culpable"),
    ("Drug", r"cocaine|cannabis|heroin|meth|oxycod|hydrocod|hydromorph|alprazolam|clonazepam|diazepam|lorazepam"
             r"|amphet|morphine|codeine|fentanyl|methadone|bupren|carisoprodol|phentermine|oxymorph|mdma|lsd"
             r"|ecstasy|ethylone|butylone|xlr11|pyrrolid|piperaz|pipe\b|controlled? subst|contr sub|control subst"
             r"|cont subs|paraphernalia|tetrahydro|\brx\b|prescript|drug|traff|jwh|fluoro|gamma-hydroxy|butanediol"
             r"|steroid|trenbolon|amobarbital|methox|contrft schd|countrft|cntrft|canbs|fubinaca|synthetic"),
    ("Weapon", r"firearm|weapon|\bwep\b|f/arm|concealed|ammun|destruct device|bomb"),
    ("Fraud", r"forg|utter|fraud|credit card|\bcc\b|personal id|identity|false name|false own|counterfeit|worthless"
              r"|defraud|launder|structuring|scanning device|innkeeper|ID card|unlaw.*\bid\b|unlaw use false"
              r"|imperson|police badge|simulation|draft|trade secret|intellectual|bribery|telemarket"
              r"|white collar|bad check|false info|false 911|false crime|giving false|posses.*altered bills|unlaw pos of prson|hire|lease"),
    ("Property", r"theft|burgl|stolen|mischief|shoplift|arson|trespass|tresspass|retail|cart|tools|fire extin|\bburg\b|burg/"
                 r"|altered id|alt vin|railroad|fuel|deprive"),
]
COMPILED = [(n, re.compile(rx, re.I)) for n, rx in RULES]


def categorize(desc):
    if pd.isna(desc) or not str(desc).strip():
        return "Unknown"
    for name, rx in COMPILED:
        if rx.search(str(desc)):
            return name
    return "Other"


for s in ["Battery", "Possession of Cocaine", "Driving While License Revoked", "Petit Theft", "Resist/obstruct Officer W/viol"]:
    print(f"{s:35s} -> {categorize(s)}")

Battery                             -> Violent
Possession of Cocaine               -> Drug
Driving While License Revoked       -> Traffic
Petit Theft                         -> Property
Resist/obstruct Officer W/viol      -> Other


## Шаг 4. Заключения (custody)

1. Берём только людей, у которых есть арест в `casearrest`.
2. Оставляем заключения с датой входа не раньше **2013-01-01**: `jailhistory` в базе есть только с 2013 года,
   а более ранние записи `prisonhistory` дали бы ложные многолетние промежутки между заключениями.
3. Сортируем по человеку и **точному** времени входа `in_t` (со временем суток), затем округляем вход до дня.
4. Убираем точные дубли (тот же человек, тот же день входа и выхода).
5. Подклеиваем `age, sex, race` из `people` и переименовываем `person_id` → `id`.

In [10]:
ids = set(arrests["person_id"])
hist["in_t"] = pd.to_datetime(hist["in_custody"], format="mixed")
hist["in_custody"] = hist["in_t"].dt.normalize()
hist["out_custody"] = day(hist["out_custody"])

cust = (hist[hist["person_id"].isin(ids) & (hist["in_custody"] >= START)]
        .sort_values(["person_id", "in_t", "out_custody"])
        .drop_duplicates(["person_id", "in_custody", "out_custody"])
        .reset_index(drop=True))
cust = cust.merge(people, on="person_id", how="left")
cust = cust.rename(columns={"person_id": "id"})
print("людей:", cust["id"].nunique(), "| заключений:", len(cust))
print(cust["custody_source"].value_counts().to_string())

людей: 10580 | заключений: 22552
custody_source
jail      21417
prison     1135


## Шаг 5. Производные столбцы по человеку (как у Даши)

Для каждого человека заключения идут по порядку, и для каждого заключения считаем:

| Столбец | Формула |
|---|---|
| `custody_num` | номер заключения у человека (1, 2, …) |
| `time_to_last_custody` | `in_custody` − предыдущий `in_custody`, дни |
| `time_since_last_arrest` | `in_custody` − предыдущий `out_custody`, дни (время на свободе) |
| `time_to_future_arrest` | следующий `in_custody` − `in_custody`, дни; пусто у последнего |
| `event`, `event_of_future_arrest` | 1, если есть следующее заключение; 0 у последнего (цензура) |
| `curr_dur_custody` | `out_custody` − `in_custody`, дни |
| `sum_dur_custody` | сумма длительностей **предыдущих** заключений |
| `average_dur_custody` | `sum_dur_custody / (custody_num − 1)`, 2 знака; пусто у первого |
| `all_future_arrest_times` | список дней от `in_custody` до каждого будущего входа |
| `all_future_events` | `[1]*число будущих заключений + [0]` |

Столбец `ep` — сквозной номер заключения в таблице, по нему потом подклеим обвинения.

In [11]:
def lst(v):
    return "[" + ", ".join(str(int(x)) for x in v) + "]"


def timeline(g):
    i, o, n = g["in_custody"], g["out_custody"], len(g)
    r = pd.DataFrame(index=g.index)
    r["custody_num"] = np.arange(1, n + 1)
    r["time_to_last_custody"] = (i - i.shift()).dt.days
    r["time_since_last_arrest"] = (i - o.shift()).dt.days
    r["time_to_future_arrest"] = (i.shift(-1) - i).dt.days
    r["event"] = (np.arange(n) < n - 1).astype(int)
    r["event_of_future_arrest"] = r["event"]
    dur = (o - i).dt.days
    r["curr_dur_custody"] = dur
    r["sum_dur_custody"] = dur.fillna(0).cumsum().shift(fill_value=0).astype(int)
    avg = r["sum_dur_custody"] / (r["custody_num"] - 1).replace(0, np.nan)
    r["average_dur_custody"] = avg.round(2)
    days = [(i.iloc[k + 1:] - i.iloc[k]).dt.days.tolist() for k in range(n)]
    r["all_future_arrest_times"] = [lst(d) if d else np.nan for d in days]
    r["all_future_events"] = [lst([1] * len(d) + [0]) for d in days]
    return r


cust = pd.concat([cust, pd.concat([timeline(g) for _, g in cust.groupby("id", sort=False)])], axis=1)
cust["ep"] = np.arange(len(cust))
cust.head()

,id,in_custody,out_custody,custody_source,in_t,age,sex,race,custody_num,time_to_last_custody,time_since_last_arrest,time_to_future_arrest,event,event_of_future_arrest,curr_dur_custody,sum_dur_custody,average_dur_custody,all_future_arrest_times,all_future_events,ep
0,1,2013-08-13,2013-08-14,jail,2013-08-13 06:03:42,69,Male,Other,1,NaN,NaN,328.0,1,1,1,0,NaN,[328],"[1, 0]",0
1,1,2014-07-07,2014-07-14,jail,2014-07-07 09:26:12,69,Male,Other,2,328.0,327.0,NaN,0,0,7,1,1.0,NaN,[0],1
2,3,2013-01-26,2013-02-05,jail,2013-01-26 03:45:27,34,Male,African-American,1,NaN,NaN,992.0,1,1,10,0,NaN,[992],"[1, 0]",2
3,3,2015-10-15,2015-12-07,prison,2015-10-15 00:00:00,34,Male,African-American,2,992.0,982.0,NaN,0,0,53,10,10.0,NaN,[0],3
4,4,2013-04-13,2013-04-14,jail,2013-04-13 04:58:34,24,Male,African-American,1,NaN,NaN,64.0,1,1,1,0,NaN,"[64, 108, 348, 1000]","[1, 1, 1, 1, 0]",4


## Шаг 6. Обвинения каждого заключения

Проблема: структура таблицы арестов (prison/jail) - id / name / in-out custody / person_id, нет инфы о номере дела
=> номер дела привязывается по правилам ниже 

В базе нет прямой ссылки «заключение → дело», поэтому связываем по датам. Для заключения с датой входа `d`
правила проверяются по очереди, берётся первое сработавшее (`charge_source`):

1. **`arrest`** — арест в `casearrest` с датой в `[d − 1, d + 1]` дней; берём все обвинения этого дела (или дел).
2. **`offense`** — ареста нет, но в `charge` есть обвинения с датой преступления `offense_date` в `[d − 7, d + 1]`.
3. **`prior_case`** — нет и этого (обычно приговор или нарушение условий по старому делу): берём обвинения
   **последнего** ареста за 365 дней до входа.
4. **`none`** — ничего не нашлось: одна строка с пустым обвинением.

Внутри заключения повторы (то же дело и тот же номер обвинения) убираются, порядок — по `case_number`, `charge_number`.
Цикл идёт по ~20 тыс. заключений, занимает несколько секунд.

In [17]:
a_by = dict(tuple(arr_ch.groupby("person_id")))   # аресты с обвинениями, по человеку
c_by = dict(tuple(charges.groupby("person_id")))  # все обвинения, по человеку
one = pd.Timedelta(days=1)
rows = []
for ep, pid, d in zip(cust["ep"], cust["id"], cust["in_custody"]):
    a = a_by.get(pid)
    found, src = None, "none"
    # 1) арест в окне ±1 день
    if a is not None:
        x = a[(a["arrest_date"] >= d - one) & (a["arrest_date"] <= d + one)]
        if len(x):
            found, src = x, "arrest"
    # 2) дата преступления за 7 дней до входа (или на следующий день)
    if found is None and pid in c_by:
        c = c_by[pid]
        x = c[(c["offense_date"] >= d - 7 * one) & (c["offense_date"] <= d + one)]
        if len(x):
            found, src = x, "offense"
    # 3) последнее дело за год до входа
    if found is None and a is not None:
        x = a[(a["arrest_date"] < d - one) & (a["arrest_date"] >= d - 365 * one)]
        if len(x):
            found, src = x[x["arrest_date"] == x["arrest_date"].max()], "prior_case"
    if found is not None:
        f = (found.drop_duplicates(["case_number", "charge_number"])
             .sort_values(["case_number", "charge_number"]))
        rows.append(pd.DataFrame({"ep": ep, "c_charge_degree": f["charge_degree"].values,
                                  "c_charge_desc": f["charge"].values,
                                  "case_number": f["case_number"].values, "charge_source": src}))
    else:
        # 4) ничего не нашли
        rows.append(pd.DataFrame({"ep": [ep], "charge_source": ["none"]}))

ch = pd.concat(rows, ignore_index=True)
print("строк обвинений:", len(ch))
print(ch.drop_duplicates("ep")["charge_source"].value_counts().to_string())

строк обвинений: 65363
charge_source
arrest        20368
prior_case     1335
none            778
offense          71


## Шаг 7. Номер обвинения, число обвинений, категория

- `n_charges` — сколько обвинений у заключения (0, если `none`);
- `charge_num` — номер обвинения внутри заключения (1, 2, …; 0, если обвинений нет);
- `charge_category` — категория из шага 3.

In [13]:
ch["n_charges"] = ch.groupby("ep")["c_charge_desc"].transform("count")
ch["charge_num"] = np.where(ch["n_charges"] > 0, ch.groupby("ep").cumcount() + 1, 0)
ch["charge_category"] = ch["c_charge_desc"].map(categorize)
ch["charge_category"].value_counts()

charge_category
Traffic     16147
Property    13410
Drug        12907
Violent      9933
Other        6123
Fraud        3748
Sex          1396
Weapon        921
Unknown       778
Name: count, dtype: int64

## Шаг 8. Итоговая таблица

Соединяем заключения (шаги 4–5) с обвинениями (шаги 6–7) по `ep`, сортируем по человеку, номеру заключения
и номеру обвинения, даты пишем как `ГГГГ-ММ-ДД`. Порядок столбцов — как у Даши, новые столбцы в конце.

In [14]:
out = cust.merge(ch, on="ep").sort_values(["id", "custody_num", "charge_num"])
for c in ["in_custody", "out_custody"]:
    out[c] = out[c].dt.strftime("%Y-%m-%d")
cols = ["id", "age", "sex", "race", "c_charge_degree", "c_charge_desc", "in_custody", "out_custody",
        "custody_num", "time_to_last_custody", "time_since_last_arrest", "time_to_future_arrest", "event",
        "event_of_future_arrest", "curr_dur_custody", "sum_dur_custody", "average_dur_custody",
        "all_future_arrest_times", "all_future_events",
        "charge_num", "n_charges", "charge_category", "charge_source", "case_number", "custody_source"]
out = out[cols]
print("строк (обвинений):", len(out), "| заключений:", out[["id", "custody_num"]].drop_duplicates().shape[0],
      "| людей:", out["id"].nunique())
out.head(10)

строк (обвинений): 65363 | заключений: 22552 | людей: 10580


,id,age,sex,race,c_charge_degree,c_charge_desc,in_custody,out_custody,custody_num,time_to_last_custody,...,sum_dur_custody,average_dur_custody,all_future_arrest_times,all_future_events,charge_num,n_charges,charge_category,charge_source,case_number,custody_source
0,1,69,Male,Other,(F3),Aggravated Assault w/Firearm,2013-08-13,2013-08-14,1,NaN,...,0,NaN,[328],"[1, 0]",1,2,Violent,arrest,13011352CF10A,jail
1,1,69,Male,Other,(M1),Battery,2013-08-13,2013-08-14,1,NaN,...,0,NaN,[328],"[1, 0]",2,2,Violent,arrest,13011352CF10A,jail
2,1,69,Male,Other,(F3),Aggravated Assault w/Firearm,2014-07-07,2014-07-14,2,328.0,...,1,1.0,NaN,[0],1,2,Violent,arrest,13011352CF10A,jail
3,1,69,Male,Other,(M1),Battery,2014-07-07,2014-07-14,2,328.0,...,1,1.0,NaN,[0],2,2,Violent,arrest,13011352CF10A,jail
4,3,34,Male,African-American,(F3),Felony Battery w/Prior Convict,2013-01-26,2013-02-05,1,NaN,...,0,NaN,[992],"[1, 0]",1,1,Violent,arrest,13001275CF10A,jail
5,3,34,Male,African-American,NaN,NaN,2015-10-15,2015-12-07,2,992.0,...,10,10.0,NaN,[0],0,0,Unknown,none,NaN,prison
6,4,24,Male,African-American,(M1),Resist/Obstruct W/O Violence,2013-04-13,2013-04-14,1,NaN,...,0,NaN,"[64, 108, 348, 1000]","[1, 1, 1, 1, 0]",1,4,Other,arrest,13002653MM10A,jail
7,4,24,Male,African-American,(F3),Possession of Cocaine,2013-04-13,2013-04-14,1,NaN,...,0,NaN,"[64, 108, 348, 1000]","[1, 1, 1, 1, 0]",2,4,Drug,arrest,13005330CF10A,jail
8,4,24,Male,African-American,(M1),Resist/Obstruct W/O Violence,2013-04-13,2013-04-14,1,NaN,...,0,NaN,"[64, 108, 348, 1000]","[1, 1, 1, 1, 0]",3,4,Other,arrest,13005330CF10A,jail
9,4,24,Male,African-American,(M1),Possess Drug Paraphernalia,2013-04-13,2013-04-14,1,NaN,...,0,NaN,"[64, 108, 348, 1000]","[1, 1, 1, 1, 0]",4,4,Drug,arrest,13005330CF10A,jail


## Шаг 9. Сохраняем

`out/compas_custody_charges.csv` (и `.xlsx`, если `WRITE_XLSX = True`).

In [15]:
os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, "compas_custody_charges.csv")
out.to_csv(csv_path, index=False)
print("записано:", os.path.abspath(csv_path))
if WRITE_XLSX:
    try:
        with pd.ExcelWriter(os.path.join(OUT_DIR, "compas_custody_charges.xlsx"), engine="openpyxl") as w:
            out.to_excel(w, sheet_name="data", index=False)
            w.sheets["data"].freeze_panes = "B2"
        print("записано:", os.path.abspath(os.path.join(OUT_DIR, "compas_custody_charges.xlsx")))
    except ImportError:
        print("openpyxl не установлен — xlsx не записан, есть только csv")

записано: D:\diploma\jupyter lab\out\compas_custody_charges.csv
записано: D:\diploma\jupyter lab\out\compas_custody_charges.xlsx


## Дальнейший план работ

### Проблема петель

Петля (self-loop) — переход из состояния в то же самое состояние: человек снова попадает в заключение с тем же типом преступления.
В варианте «все обвинения новые» (состояние = набор категорий обвинений заключения) из **11 136** переходов между заключениями **4 935 (44%)** — петли.
Петли забивают граф: самые частые дуги ведут из вершины в неё же, а реальные переходы между типами преступлений теряются.

Главная причина — **старые дела**. В `compas.db` у дела (`case_number`) один постоянный список обвинений в таблице `charge`.
Если человека снова задерживают по уже открытому делу (суд, приговор, нарушение условий), то через правило `arrest` или `prior_case`
к новому заключению подтягиваются **те же самые обвинения**, и получается петля, хотя нового преступления не было.
Из 4 935 петель:

| Тип петли | Петель | Доля |
|---|---|---|
| в новом заключении только старые дела | 3 226 | 65% |
| в новом заключении только новые дела (реальный повтор того же типа) | 1 212 | 25% |
| смешанные (и старые, и новые дела) | 497 | 10% |

Старым считается обвинение, `case_number` которого уже встречался в одном из **предыдущих** заключений того же `id`.

### Варианты решения

| Вариант | Правило | Петель | Минус |
|---|---|---|---|
| Все новые | старые обвинения считаются как новые | 4 935 из 11 136 (44%) | петли из-за старых дел |
| OldCase | если **все** обвинения старые — одно состояние `OldCase`; иначе старые обвинения выбрасываются | 2 387 | теряется, по какому типу было старое дело; `OldCase → OldCase` = 993 петли |
| Без OldCase | заключения только со старыми делами удаляются из логов | 1 537 из 7 066 | пропадает треть переходов, появляются «мосты» через удалённые заключения (862) |
| Склейка | повторный вход того же типа в течение 1 дня склеивается с предыдущим | — | убирает лишь ~100 пар, проблему старых дел не решает |

### Текущий вариант (проверяется первым): все категории × 2

В уже посчитанных вариантах OldCase — **одно общее** состояние для всех старых дел. Теперь OldCase делится по категориям: `Sex → OldCase: Sex`, `Violent → OldCase: Violent` и т.д.

Каждая категория преступления существует в двух видах:

- **`Категория`** — новое обвинение (его дело раньше у этого человека не встречалось);
- **`OldCase: Категория`** — обвинение из дела, которое уже было в одном из предыдущих заключений.

Логика: если в рамках того же дела снова появляется то же обвинение, это не новое преступление, а продолжение старого дела.
Поэтому переход идёт не в петлю `Violent → Violent`, а в состояние `OldCase: Violent` (аналогично `Sex → OldCase: Sex` и т.д.). Так граф не запутывается петлями,
но, в отличие от варианта OldCase, сохраняется информация, **какого типа** было старое дело, и ни одно заключение не удаляется.
Число состояний: 15 категорий × 2 = до 30 (часть комбинаций будет редкой).

### Поэтапный план

1. **Таблица заключений с обвинениями** — этот ноутбук (`compas_custody_charges`). ✔
2. **Таблица логов (event log)** для варианта «категории × 2»: `case:concept:name` = `id`, `concept:name` = состояние
   (категория или «OldCase: категория»), `time:timestamp` = `in_custody`. Одна строка = одна категория в одном заключении;
   заключения со смешанными Justice и другими обвинениями делятся на два шага; последнее заключение человека — цензурировано (без END).
3. **Статистика логов**: число состояний, переходов и петель, частые варианты трасс; сравнение с вариантами «все новые», OldCase и «без OldCase».
4. **Построение графа** в PM4Py: DFG (частоты и времена), Heuristics Miner, Inductive Miner → сети Петри; один общий граф по трассам всех людей.
5. **Оценка моделей**: fitness и precision на тестовой выборке людей, выбор модели и порога шума.
6. **Матрица переходов**: вероятности переходов между состояниями и время до следующего заключения по каждой дуге.
7. **Модель следующего события**: конкурирующие риски с цензурированием (куда и когда перейдёт человек),
   ковариаты — возраст, пол, раса, степень обвинения, длительности заключений.
8. **Перенос на Instacart**: состояние = сегмент покупателя (кластеризация).